In [75]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, SpectralClustering
import numpy as np
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [76]:
df_train = pd.read_csv("data/fashion-mnist_train.csv")
df_train.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,6,0,0,0,0,0,0,0,5,0,...,0,0,0,30,43,0,0,0,0,0
3,0,0,0,0,1,2,0,0,0,0,...,3,0,0,0,0,1,0,0,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [77]:
df_test = pd.read_csv("data/fashion-mnist_test.csv")
df_test.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,0,0,0,0,0,0,0,0,9,8,...,103,87,56,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,...,34,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,14,53,99,...,0,0,0,0,63,53,31,0,0,0
3,2,0,0,0,0,0,0,0,0,0,...,137,126,140,0,133,224,222,56,0,0
4,3,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [78]:
y_train = df_train['label'].values
X_train = df_train.drop(columns=['label']).values


In [79]:
y_test = df_test['label'].values
X_test = df_test.drop(columns=['label']).values

## Preprocessing

In [82]:
# scaling data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [83]:
# Dimensionality reduction
pca = PCA(n_components=0.98, random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
print("Original dimensions:", X_train.shape[1])
print("Reduced dimensions:", X_train_pca.shape[1])

Original dimensions: 784
Reduced dimensions: 420


In [84]:
X_train_pca

array([[ 1.06649729e+01,  1.49933635e+01, -6.89468090e-01, ...,
         1.94650873e-01,  2.90525883e-01,  2.00916996e-01],
       [-1.19897476e+01,  1.18127701e+01, -5.80104873e+00, ...,
        -8.91169666e-02, -2.66695694e-01,  2.36396297e-01],
       [ 2.05176712e+01,  1.57978434e+00,  6.77012222e+00, ...,
        -8.68449390e-02, -3.42630897e-01,  6.50458273e-02],
       ...,
       [ 7.14876699e+00, -5.43564927e-01, -8.74059571e-01, ...,
        -2.34760325e-01,  3.74893492e-01,  2.18045840e-01],
       [ 4.43415256e+00,  2.08327043e+01, -5.76302113e-01, ...,
         4.16061527e-01,  1.16798892e-02,  5.84275712e-01],
       [-9.10680985e+00,  1.48311538e+01, -3.67467806e+00, ...,
         2.16371733e-01, -2.72634847e-01,  1.30955424e-01]])

In [85]:
X_test_pca  = pca.transform(X_test_scaled)

## 1. Data Processing and Unsupervised Learning.

**(a) Compute and report a frequency table of the outcome variable in both training and
testing data.**

In [44]:
train_freq = df_train['label'].value_counts()

In [45]:
test_freq = df_test['label'].value_counts()

In [46]:
freq_table = pd.DataFrame({
    'Train Count': train_freq,
    'Test Count': test_freq
})
freq_table['Train %'] = (freq_table['Train Count'] / train_freq.sum() * 100).round(2)
freq_table['Test %']  = (freq_table['Test Count'] / test_freq.sum() * 100).round(2)
freq_table

,Train Count,Test Count,Train %,Test %
0,6000,1000,10.0,10.0
1,6000,1000,10.0,10.0
2,6000,1000,10.0,10.0
3,6000,1000,10.0,10.0
4,6000,1000,10.0,10.0
5,6000,1000,10.0,10.0
6,6000,1000,10.0,10.0
7,6000,1000,10.0,10.0
8,6000,1000,10.0,10.0
9,6000,1000,10.0,10.0


**(b) Perform two clustering algorithms to the training data.
– Use a systematic approach (e.g., gap statistic, silhouette statistic) to find the
optimal number of clusters (if applicable).
– What is the dominating label in each of these clusters?
– Do your clusters help to separate the labels? Comment on the results**

In [47]:
# K-means clustering
k_range = range(4, 12)
kmeans_scores = {}
sample_size = 10000
for k in k_range:
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = kmeans.fit_predict(X_train_pca)
    
    score = silhouette_score(
            X_train_pca, labels,
            sample_size=min(sample_size, X_train_pca.shape[0]),
        )
    
    kmeans_scores[k] = score
    print(f"KMeans: k={k}, silhouette={score:.4f}")

    
best_k_kmeans = max(kmeans_scores, key=kmeans_scores.get)
print("Best k for KMeans:", best_k_kmeans)

KMeans: k=4, silhouette=0.1472
KMeans: k=5, silhouette=0.1232
KMeans: k=6, silhouette=0.1264
KMeans: k=7, silhouette=0.1364
KMeans: k=8, silhouette=0.1411
KMeans: k=9, silhouette=0.1414
KMeans: k=10, silhouette=0.1304
KMeans: k=11, silhouette=0.1276
Best k for KMeans: 4


In [48]:
# best_k_kmeans = 10
kmeans_final = KMeans(n_clusters=best_k_kmeans, n_init=10, random_state=42)
kmeans_labels = kmeans_final.fit_predict(X_train_pca)

In [49]:
# dominant label
cluster_df_kmeans = pd.DataFrame({
    'cluster': kmeans_labels,
    'label': y_train
})
# Frequency table: cluster × true label
crosstab_kmeans = pd.crosstab(cluster_df_kmeans['cluster'], cluster_df_kmeans['label'])
print(crosstab_kmeans)

# Dominant label per cluster
dominant_label_kmeans = crosstab_kmeans.idxmax(axis=1)
dominant_count_kmeans = crosstab_kmeans.max(axis=1)

dominant_label_kmeans
summary_kmeans = pd.DataFrame({
    'cluster': crosstab_kmeans.index,
    'dominant_label': dominant_label_kmeans,
    'dominant_count': dominant_count_kmeans,
    'cluster_size': crosstab_kmeans.sum(axis=1),
})

summary_kmeans['purity'] = summary_kmeans['dominant_count'] / summary_kmeans['cluster_size']
# print(summary_kmeans.sort_values('cluster'))
summary_kmeans

label       0     1     2     3     4     5     6     7     8     9
cluster                                                            
0        2454  5723   221  5047   850     8  1028     0   406    41
1          23     6    56     1    19   465    80   546  3401  5270
2         925   158  1403   531   608  5525  1625  5454  1165   625
3        2598   113  4320   421  4523     2  3267     0  1028    64


,cluster,dominant_label,dominant_count,cluster_size,purity
cluster,,,,,
0,0,1,5723,15778,0.362720
1,1,9,5270,9867,0.534104
2,2,5,5525,18019,0.306621
3,3,4,4523,16336,0.276873


In [53]:
# Another clustering algo (need to choose from Hierarchical clustering, Self-Organizing Maps, or Spectral clustering since those are the ones discussed in lecture)
subset_size = 10000
rng = np.random.RandomState(42)
indices = rng.choice(X_train_pca.shape[0], size=subset_size, replace=False)

X_spec = X_train_pca[indices]
y_spec = y_train[indices]

k_range = range(4, 12)
spec_scores = {}
sample_size = 10000
n_neighbors = 10
for k in k_range:
    spec = SpectralClustering(
            n_clusters=k,
            affinity='nearest_neighbors',
            n_neighbors=n_neighbors,
            assign_labels='kmeans',
            n_init=10,
        )
    labels = spec.fit_predict(X_spec)
    
    score = silhouette_score(
            X_spec, labels,
            sample_size=min(sample_size, X_spec.shape[0]),
        )
    
    spec_scores[k] = score
    print(f"Spectral: k={k}, silhouette={score:.4f}")

    
best_k_spectral = max(spec_scores, key=spec_scores.get)
print("Best k for Spectral:", best_k_spectral)

Spectral: k=4, silhouette=0.0679
Spectral: k=5, silhouette=0.0827
Spectral: k=6, silhouette=0.1007
Spectral: k=7, silhouette=0.1000
Spectral: k=8, silhouette=0.1059
Spectral: k=9, silhouette=0.1128
Spectral: k=10, silhouette=0.1208
Spectral: k=11, silhouette=0.1165
Best k for Spectral: 10


In [54]:
spec_final = SpectralClustering(
    n_clusters=best_k_spectral,
    affinity='nearest_neighbors',
    n_neighbors=10,
    assign_labels='kmeans',
    n_init=10,
    random_state=42
)

spec_labels = spec_final.fit_predict(X_spec)

In [55]:
# dominant label
cluster_df_spectral = pd.DataFrame({
    'cluster': spec_labels,
    'label': y_spec
})
# Frequency table: cluster × true label
crosstab_spectral = pd.crosstab(cluster_df_spectral['cluster'], cluster_df_spectral['label'])
print(crosstab_spectral)

# Dominant label per cluster
dominant_label_spectral = crosstab_spectral.idxmax(axis=1)
dominant_count_spectral = crosstab_spectral.max(axis=1)

dominant_label_kmeans
summary_spectral = pd.DataFrame({
    'cluster': crosstab_spectral.index,
    'dominant_label': dominant_label_spectral,
    'dominant_count': dominant_count_spectral,
    'cluster_size': crosstab_spectral.sum(axis=1),
})

summary_spectral['purity'] = summary_spectral['dominant_count'] / summary_spectral['cluster_size']
summary_spectral

label      0    1    2    3    4    5    6    7    8    9
cluster                                                  
0          0    0    0    1    0   70    1  178   10  620
1          4    0    5    0    0    0    6    0  387    0
2         26   19  748    9  687    0  447    0   23    0
3        241   59  226  305  100  306  326    1   76    7
4          0  869    0    0    1    0    0    0    0    0
5          0    0    1    0    0  531    1  818   12   22
6          0    0    0    0    1    0    1    0  432    0
7        627    0    3    9    1    0  168    0    1    0
8          0    0    0    0    0   39    0    2    3  371
9        134   27   18  713  202    0   97    0    8    0


,cluster,dominant_label,dominant_count,cluster_size,purity
cluster,,,,,
0,0,9,620,880,0.704545
1,1,8,387,402,0.962687
2,2,2,748,1959,0.381827
3,3,6,326,1647,0.197936
4,4,1,869,870,0.998851
5,5,7,818,1385,0.590614
6,6,8,432,434,0.995392
7,7,0,627,809,0.775031
8,8,9,371,415,0.893976


KMeans seems to perform worse than spectral. We know that there should be 10 clusters overall from the dataset. However, the optimal K using sillhoutte score for KMeans is 4, whereas for spectral clustering it is indeed 10. If we look at the purity for each cluster as well, the clusters from spectral clustering are more pure

## 2. Multi-class Classification Model

**Choose four of the methods we discussed in class to do multi-class classification. Similar to coding assignment 4, some of the methods were presented for binary classification,
but you are free to extend and use them for multi-class classification (e.g. you can use
One-vs-Rest). Make sure you describe your approach.**

**Report the overall classification error. Also, please provide enough supporting information, e.g. tables/figures, to demonstrate the fit of the models.**

Method 1: Random Tree (Decision Tree)

In [67]:
dt = DecisionTreeClassifier(
    criterion='gini',      # or 'entropy'
    max_depth=None,        # let it grow fully → likely overfits
    random_state=42
)

dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

print("Decision Tree accuracy:", accuracy_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

Decision Tree accuracy: 0.7963
              precision    recall  f1-score   support

           0       0.73      0.73      0.73      1000
           1       0.94      0.96      0.95      1000
           2       0.69      0.68      0.69      1000
           3       0.82      0.81      0.82      1000
           4       0.68      0.69      0.68      1000
           5       0.90      0.87      0.88      1000
           6       0.55      0.56      0.56      1000
           7       0.85      0.86      0.86      1000
           8       0.91      0.90      0.91      1000
           9       0.88      0.90      0.89      1000

    accuracy                           0.80     10000
   macro avg       0.80      0.80      0.80     10000
weighted avg       0.80      0.80      0.80     10000



Method 2: Random Tree with Boosting (AdaBoost)

In [72]:
base_tree = DecisionTreeClassifier(
    max_depth=2,          
    random_state=42
)

ada = AdaBoostClassifier(
    base_estimator=base_tree,
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

ada.fit(X_train, y_train)
y_pred_ada = ada.predict(X_test)

print("AdaBoost (tree) accuracy:", accuracy_score(y_test, y_pred_ada))
print(classification_report(y_test, y_pred_ada))

AdaBoost (tree) accuracy: 0.6384
              precision    recall  f1-score   support

           0       0.67      0.27      0.39      1000
           1       0.89      0.87      0.88      1000
           2       0.51      0.73      0.60      1000
           3       0.65      0.75      0.70      1000
           4       0.52      0.58      0.55      1000
           5       0.53      0.88      0.66      1000
           6       0.33      0.30      0.31      1000
           7       0.82      0.57      0.68      1000
           8       0.83      0.94      0.88      1000
           9       0.96      0.48      0.64      1000

    accuracy                           0.64     10000
   macro avg       0.67      0.64      0.63     10000
weighted avg       0.67      0.64      0.63     10000



Method 3: Random Forrest Classifier

In [73]:
rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=None,       
    max_features='sqrt',  
    n_jobs=-1,            
    random_state=42
)

rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("Random Forest accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest accuracy: 0.8847
              precision    recall  f1-score   support

           0       0.82      0.86      0.84      1000
           1       0.99      0.97      0.98      1000
           2       0.80      0.80      0.80      1000
           3       0.89      0.94      0.91      1000
           4       0.80      0.86      0.83      1000
           5       0.98      0.95      0.96      1000
           6       0.75      0.61      0.67      1000
           7       0.92      0.94      0.93      1000
           8       0.95      0.97      0.96      1000
           9       0.94      0.95      0.94      1000

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



Method 4: KNN

In [86]:
for k in range(3,10):
    knn = KNeighborsClassifier(n_neighbors=k, weights='distance')
    knn.fit(X_train_pca, y_train)
    acc = accuracy_score(y_test, knn.predict(X_test_pca))
    print(f"k={k}: accuracy={acc:.4f}")

k=3: accuracy=0.8617
k=4: accuracy=0.8684
k=5: accuracy=0.8684
k=6: accuracy=0.8696
k=7: accuracy=0.8655
k=8: accuracy=0.8693
k=9: accuracy=0.8642


In [88]:
best_k = 6 # hardcoded based on output of above cell

In [89]:
knn = KNeighborsClassifier(
    n_neighbors=best_k,
    weights='distance',
    metric='euclidean'
)

knn.fit(X_train_pca, y_train)
y_pred_knn = knn.predict(X_test_pca)

print("k-NN accuracy:", accuracy_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn))

k-NN accuracy: 0.8696
              precision    recall  f1-score   support

           0       0.80      0.86      0.83      1000
           1       0.99      0.97      0.98      1000
           2       0.80      0.78      0.79      1000
           3       0.91      0.89      0.90      1000
           4       0.80      0.82      0.81      1000
           5       0.99      0.86      0.92      1000
           6       0.66      0.66      0.66      1000
           7       0.89      0.94      0.92      1000
           8       0.98      0.95      0.97      1000
           9       0.90      0.96      0.93      1000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000

